In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import textstat

df = pd.read_csv('../output/conv_item_loadings_rotated_geomin_obl.csv')

In [2]:
from collections import defaultdict

def cap_by_type(df_in, type_col='type', max_per_type=25):
    # preserve the original row order while limiting each type to max_per_type
    counts = defaultdict(int)
    keep_idx = []
    for idx, t in zip(df_in.index, df_in[type_col]):
        if counts[t] < max_per_type:
            keep_idx.append(idx)
            counts[t] += 1
    return df_in.loc[keep_idx]

# F1 Analysis

In [3]:
top_f1 = df.sort_values('Factor_1', ascending=False).head(1000)
bottom_f1 = df.sort_values('Factor_1', ascending=True).head(1000)


top_f1 = cap_by_type(top_f1)
bottom_f1 = cap_by_type(bottom_f1)

print(top_f1['type'].value_counts())
print(bottom_f1['type'].value_counts())

type
international_citizenship       25
function_of_decision_section    25
corporate_lobbying              25
abercrombie                     14
proa                            14
Name: count, dtype: int64
type
international_citizenship       25
corporate_lobbying              25
proa                            25
abercrombie                     25
function_of_decision_section    25
Name: count, dtype: int64


In [4]:
print(top_f1['answer'].value_counts())
print(bottom_f1['answer'].value_counts())

answer
No                    62
Procedural History     7
Issue                  6
suggestive             6
Rule                   5
descriptive            4
Conclusion             3
Analysis               3
Yes                    2
generic                2
arbitrary              2
Facts                  1
Name: count, dtype: int64
answer
Yes                   74
fanciful              11
generic                7
Facts                  6
Analysis               6
Rule                   4
arbitrary              3
Procedural History     3
descriptive            3
Issue                  3
Conclusion             2
No                     1
suggestive             1
Decree                 1
Name: count, dtype: int64


In [5]:
# display(top_f1['question'].tolist())
# print('=====')
# display(bottom_f1['question'].tolist())

# F2 analysis

In [6]:
top_f2 = df.sort_values('Factor_2', ascending=False).head(1000)
bottom_f2 = df.sort_values('Factor_2', ascending=True).head(1000)

top_f2 = cap_by_type(top_f2)
bottom_f2 = cap_by_type(bottom_f2)

print(top_f2['type'].value_counts())
print(bottom_f2['type'].value_counts())

type
corporate_lobbying              25
international_citizenship       25
proa                            25
function_of_decision_section    25
abercrombie                     25
Name: count, dtype: int64
type
international_citizenship       25
function_of_decision_section    25
abercrombie                     25
corporate_lobbying              25
proa                            17
Name: count, dtype: int64


In [7]:
print(top_f2['answer'].value_counts())
print(bottom_f2['answer'].value_counts())

answer
No                    50
Yes                   25
generic               11
fanciful               9
Facts                  6
Rule                   5
Analysis               5
Decree                 4
Conclusion             3
arbitrary              2
suggestive             2
Procedural History     1
descriptive            1
Issue                  1
Name: count, dtype: int64
answer
No                    38
Yes                   29
Issue                 14
descriptive           11
Procedural History     6
suggestive             6
Rule                   4
arbitrary              4
generic                2
fanciful               2
Conclusion             1
Name: count, dtype: int64


In [8]:
# display(top_f2['question'].tolist())
# print('=====')
# display(bottom_f2['question'].tolist())

In [ ]:
word_pattern = re.compile(r"\b\w+\b")
sentiment_analyzer = SentimentIntensityAnalyzer()

def compute_text_metrics(series, include_sentiment_label=False):
    metrics = []
    for text in series.fillna(""):
        text_str = str(text)
        tokens = word_pattern.findall(text_str.lower())
        token_lengths = [len(token) for token in tokens]
        question_length = len(tokens)
        avg_length = float(np.mean(token_lengths)) if token_lengths else 0.0
        burstiness = float(np.std(token_lengths) / avg_length) if token_lengths and avg_length else 0.0
        if token_lengths:
            counts = Counter(tokens)
            total = sum(counts.values())
            probs = np.array(list(counts.values()), dtype=float) / total
            entropy = float(-np.sum(probs * np.log(probs)))
            perplexity = float(np.exp(entropy))
        else:
            perplexity = 0.0
        flesch_kincaid = float(textstat.flesch_kincaid_grade(text_str)) if text_str.strip() else 0.0
        sentiment_scores = sentiment_analyzer.polarity_scores(text_str) if text_str.strip() else {"compound": 0.0}
        compound_sentiment = float(sentiment_scores.get("compound", 0.0))
        record = {
            "question": text_str,
            "question_length": question_length,
            "average_word_length": avg_length,
            "burstiness": burstiness,
            "perplexity": perplexity,
            "flesch_kincaid_grade": flesch_kincaid,
            "sentiment_compound": compound_sentiment
        }
        if include_sentiment_label:
            if compound_sentiment >= 0.05:
                sentiment_label = "positive"
            elif compound_sentiment <= -0.05:
                sentiment_label = "negative"
            else:
                sentiment_label = "neutral"
            record["sentiment_label"] = sentiment_label
        metrics.append(record)
    return pd.DataFrame(metrics)

In [ ]:
df_metrics = compute_text_metrics(df['question'])
top_f1_metrics = compute_text_metrics(top_f1['question'])
bottom_f1_metrics = compute_text_metrics(bottom_f1['question'])
top_f2_metrics = compute_text_metrics(top_f2['question'])
bottom_f2_metrics = compute_text_metrics(bottom_f2['question'])


summary_columns = ["question_length", "average_word_length", "burstiness", "perplexity", "flesch_kincaid_grade", "sentiment_compound"]
summary_df = pd.concat(
    [
        df_metrics[summary_columns].mean().rename("overall_mean"),
        top_f1_metrics[summary_columns].mean().rename("top_f1_mean"),
        bottom_f1_metrics[summary_columns].mean().rename("bottom_f1_mean"),
        top_f2_metrics[summary_columns].mean().rename("top_f2_mean"),
        bottom_f2_metrics[summary_columns].mean().rename("bottom_f2_mean")
    ],
    axis=1
)

display(summary_df)

for label, metrics_df in [
    ("overall", df_metrics),
    ("top_f1", top_f1_metrics),
    ("bottom_f1", bottom_f1_metrics),
    ("top_f2", top_f2_metrics),
    ("bottom_f2", bottom_f2_metrics)
]:
    print(f"\nSample metrics for {label} questions:")
    display(metrics_df.head(5))
    if "sentiment_label" in metrics_df.columns:
        print("Sentiment label distribution:")
        display(metrics_df['sentiment_label'].value_counts(normalize=True).rename(lambda x: f"{x} ({label})"))

,overall_mean,top_f1_mean,bottom_f1_mean,top_f2_mean,bottom_f2_mean
question_length,256.478217,245.757282,208.640000,227.952000,218.427350
average_word_length,5.226676,5.214247,5.111462,5.160494,5.234022
burstiness,0.568251,0.571428,0.572394,0.568643,0.565525
perplexity,72.579784,73.245803,62.189698,65.921706,62.877297
flesch_kincaid_grade,13.632242,14.437762,13.658160,14.096260,12.772313
sentiment_compound,0.207861,0.233890,0.103414,0.202448,0.154241



Sample metrics for overall questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,"Description: The mark ""7-Eleven"" for a conveni...",17,4.352941,0.649493,15.668724,6.875000,0.0000
1,"Description: The mark ""Airbus"" for an airplane...",8,6.125000,0.585469,8.000000,11.130000,0.0000
2,"Description: The mark ""Amazon"" for an online s...",9,5.555556,0.488262,9.000000,8.897778,0.1779
3,"Description: The mark ""American Airlines"" for ...",11,6.090909,0.566378,11.000000,11.227273,0.0000
4,"Description: The mark ""Antilds"" for plant seeds.",7,5.428571,0.480939,7.000000,3.997143,0.0000



Sample metrics for top_f1 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,Question: Consider the country of Moldova. Doe...,54,4.481481,0.638025,31.086778,13.951111,0.8225
1,Question: Consider the country of Iran. Does t...,33,5.030303,0.569651,26.327314,11.941970,-0.3182
2,Question: Consider the country of Kenya. Does ...,50,5.220000,0.601288,32.987698,15.400000,-0.2960
3,Question: Consider the country of Tanzania. Do...,37,4.810811,0.627958,26.652819,12.354730,0.5106
4,Question: Consider the country of Lithuania. D...,42,5.119048,0.581693,31.459373,14.565976,0.4215


...

,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,Question: Consider the country of Ethiopia. Do...,35,4.914286,0.574259,25.116646,10.789286,0.6249
1,Question: Consider the country of Angola. Does...,30,5.066667,0.555756,24.505964,9.140000,0.6249
2,Question: Consider the country of Guinea. Does...,51,5.000000,0.608437,28.675532,14.021667,-0.6486
3,Question: Consider the country of Georgia. Doe...,42,5.404762,0.562447,36.805290,15.429390,0.4215
4,Question: Consider the country of Sudan. Does ...,41,5.048780,0.699866,29.480801,13.414756,-0.8442



Sample metrics for top_f2 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,Official title of bill: A bill to provide for ...,916,5.842795,0.522386,225.346813,19.984559,0.9959
1,Official title of bill: A bill to establish a ...,1014,5.696252,0.542201,240.175435,19.261247,0.9918
2,Official title of bill: A bill to provide for ...,1064,5.408835,0.574539,233.178764,17.548594,0.9946
3,Official title of bill: A bill to reauthorize ...,983,5.215666,0.629432,191.611966,17.498836,-0.9418
4,Official title of bill: A bill to improve dive...,1006,5.761431,0.537813,209.065956,20.997707,0.9956



Sample metrics for bottom_f2 questions:


,question,question_length,average_word_length,burstiness,perplexity,flesch_kincaid_grade,sentiment_compound
0,Question: Consider the country of Namibia. Doe...,35,4.885714,0.572204,25.116646,10.452143,0.6249
1,Text: A grand jury indicted Jones for unlawful...,178,4.668539,0.509588,76.437721,13.593728,-0.9803
2,Text: Rose's complaint sought recovery of comp...,56,4.589286,0.513958,39.838984,13.669528,-0.7430
3,Question: Consider the country of Iran. Does t...,41,5.024390,0.704001,29.480801,13.414756,-0.8442
4,Text: Lassiter was a different story. Although...,21,5.857143,0.563269,19.658473,12.666905,0.0000


# Conclusion
Using oblique rotation, we get mostly similar results.
- Factor 1 strongly reflects affirmative biased questions - (e.g. "Is X true?") vs. negative biased questions + (e.g. "Is X false?")
- Factor 2 is better aligned with `Contextual Scale` - Semantic Precision, Pattern Recognition & Rule Application, High-Fidelity Textual Grounding + Abstract Reasoning, Working Memory & Attention Management, Integration with Pre-existing Knowledge